In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [2]:
dataset_path = "UTKFace"

images = []
ages = []
genders = []

files = os.listdir(dataset_path)

for img_name in files[:4000]:   # limit for faster training
    try:
        parts = img_name.split('_')
        age = int(parts[0])
        gender = int(parts[1])

        img_path = os.path.join(dataset_path, img_name)
        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.resize(img, (64, 64))

        images.append(img)
        ages.append(age)
        genders.append(gender)

    except:
        continue

images = np.array(images) / 255.0
ages = np.array(ages)
genders = np.array(genders)

print("Loaded images:", len(images))

Loaded images: 4000


In [ ]:
#  Function to group ages into categories
def get_age_group(age):
    if age < 10:
        return 0
    elif age < 20:
        return 1
    elif age < 30:
        return 2
    elif age < 40:
        return 3
    elif age < 50:
        return 4
    else:
        return 5


#  Convert lists to numpy arrays
import numpy as np
ages = np.array(ages)
genders = np.array(genders)

#  Apply age grouping
ages = np.array([get_age_group(age) for age in ages])

#  One-hot encoding
from tensorflow.keras.utils import to_categorical
ages = to_categorical(ages, 6)     # 6 age groups
genders = to_categorical(genders, 2)

# Debug prints
print("Ages shape:", ages.shape)
print("Genders shape:", genders.shape)
print("Unique age classes:", np.unique(np.argmax(ages, axis=1)))

Ages shape: (4000, 6)
Genders shape: (4000, 2)
Unique age classes: [0 1 2 5]


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense

input_layer = Input(shape=(64,64,3))

x = Conv2D(32, (3,3), activation='relu')(input_layer)
x = MaxPooling2D(2,2)(x)

x = Conv2D(64, (3,3), activation='relu')(x)
x = MaxPooling2D(2,2)(x)

x = Conv2D(128, (3,3), activation='relu')(x)
x = MaxPooling2D(2,2)(x)

x = Flatten()(x)
x = Dense(128, activation='relu')(x)

age_output = Dense(6, activation='softmax', name='age')(x)
gender_output = Dense(2, activation='softmax', name='gender')(x)

model = Model(inputs=input_layer, outputs=[age_output, gender_output])
model.compile(
    optimizer='adam',
    loss={
        'age': 'categorical_crossentropy',
        'gender': 'categorical_crossentropy'
    },
    metrics={
        'age': 'accuracy',
        'gender': 'accuracy'
    }
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 62, 62,    │        896 │ input_layer_3[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_7     │ (None, 31, 31,    │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 29, 29,    │     18,496 │ max_pooling2d_7[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_8     │ (None, 14, 14,    │          0 │ conv2d_8[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 12, 12,    │     73,856 │ max_pooling2d_8[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_9     │ (None, 6, 6, 128) │          0 │ conv2d_9[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 4608)      │          0 │ max_pooling2d_9[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │    589,952 │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ age (Dense)         │ (None, 6)         │        774 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gender (Dense)      │ (None, 2)         │        258 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 684,232 (2.61 MB)

 Trainable params: 684,232 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
print(ages.shape)
print(genders.shape)

(4000,)
(4000, 2)


In [ ]:
history = model.fit(
    images,
    {
        'age': ages,
        'gender': genders
    },
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 35ms/step - age_accuracy: 0.6209 - age_loss: 0.8482 - gender_accuracy: 0.6034 - gender_loss: 0.6534 - loss: 1.5016 - val_age_accuracy: 0.0037 - val_age_loss: 1.3906 - val_gender_accuracy: 0.7412 - val_gender_loss: 0.5497 - val_loss: 1.9402
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - age_accuracy: 0.7812 - age_loss: 0.5283 - gender_accuracy: 0.6809 - gender_loss: 0.5838 - loss: 1.1120 - val_age_accuracy: 0.3100 - val_age_loss: 1.0146 - val_gender_accuracy: 0.7825 - val_gender_loss: 0.4917 - val_loss: 1.5062
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - age_accuracy: 0.8066 - age_loss: 0.4505 - gender_accuracy: 0.7091 - gender_loss: 0.5425 - loss: 0.9930 - val_age_accuracy: 0.2788 - val_age_loss: 1.0505 - val_gender_accuracy: 0.8550 - val_gender_loss: 0.3450 - val_loss: 1.3955
Epoch 4/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - age_accuracy: 0.8213 - age_loss: 0.4007 - gender_accuracy: 0.7378 - gender_loss: 0.5048 - loss: 

In [ ]:
model.save("age_gender_model.h5")
print("Model saved successfully!")

Model saved successfully!
